# Visualise latent-displacement optimization

Load the latest persisted Bayesian-optimization snapshot and produce an easy-to-read report. This notebook can run while notebook 05 is paused or after any completed trial.

In [ ]:
ARTIFACT_DIR = None
TOP_TRIALS = 10
LOESS_FRACTION = 0.65

In [ ]:
# Explicit module reloads ensure the latest persistence and plotting code is used.

In [ ]:
from datetime import datetime
import importlib
from pathlib import Path
import sys
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "MIMIC" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
EXPERIMENT_ROOT = PROJECT_ROOT / "manuscript" / "experiments"
for path in [PROJECT_ROOT / "src", EXPERIMENT_ROOT / "src"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from streamlined import latent_displacement_tuning as tuning
from streamlined import latent_displacement_tuning_report as tuning_report
tuning = importlib.reload(tuning)
tuning_report = importlib.reload(tuning_report)

In [ ]:
artifact_dir = Path(ARTIFACT_DIR) if ARTIFACT_DIR else EXPERIMENT_ROOT / "artifacts"
live_results_path = artifact_dir / "tuning" / "latent_displacement_default_credit" / "optimization_history.csv"
result = tuning.load_tuning_result(artifact_dir)
print(f"Loaded latest snapshot: {live_results_path}")
print(f"Last updated: {datetime.fromtimestamp(live_results_path.stat().st_mtime).astimezone()}")
print(f"Completed trials: {len(result.optimization_history)}")
print(f"Best so far: {result.selected_candidate.candidate_id}")

## Results tables

In [ ]:
display(tuning_report.top_trials_table(result, n=TOP_TRIALS))
display(result.validation_summary)
if result.heldout_results.empty:
    print("Held-out evaluation is not available until all BO trials complete.")
else:
    display(tuning_report.heldout_conclusion(result))
    display(result.heldout_summary)

## Optimization progress

In [ ]:
figure, _ = tuning_report.plot_optimization_progress(result)
display(figure)
plt.close(figure)

## Pairwise hyperparameter heatmaps

In [ ]:
figure, _ = tuning_report.plot_hyperparameter_performance(result)
display(figure)
plt.close(figure)

## Marginal median and interquartile performance

In [ ]:
figure, _ = tuning_report.plot_hyperparameter_quantile_bands(
    result,
    loess_fraction=.65,
)
display(figure)
plt.close(figure)

## Save report artifacts

In [ ]:
report_manifest = tuning_report.save_optimization_report(result, artifact_dir)
display(report_manifest)